# 04 — llama.cpp Server на GPU (максимум контроля)

Нативный llama.cpp сервер с OpenAI-совместимым API. Максимальный контроль над параметрами модели.

**Требования:** Google Colab с GPU (T4/V100)  
**GPU:** ✅ Обязательно  
**API ключ:** Не нужен  
**Модель:** Gemma 3 4B GGUF (Q4_0, ~2.5GB)

In [ ]:
!nvidia-smi
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\nGPU: {gpu}, VRAM: {vram:.1f} GB")
else:
    print("⚠ GPU не доступен! Runtime → Change runtime type → T4 GPU")

# Настройки
MODEL_REPO = "google/gemma-3-4b-it-qat-q4_0-gguf"
MODEL_FILE = "gemma-3-4b-it-qat-q4_0.gguf"
N_CTX = 4096        # Размер контекста
N_GPU_LAYERS = 99   # Все слои на GPU
PORT = 8080

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import os

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    cache_dir="/content/models"
)
print(f"✓ Модель загружена: {model_path}")
print(f"  Размер: {os.path.getsize(model_path) / 1024**3:.1f} GB")

In [ ]:
# Установка llama-cpp-python с поддержкой CUDA
import os
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"

!pip install -q llama-cpp-python[server]
print("✓ llama-cpp-python установлен с CUDA")

In [ ]:
import subprocess, time, threading

cmd = [
    "python", "-m", "llama_cpp.server",
    "--model", model_path,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--n_ctx", str(N_CTX),
    "--n_gpu_layers", str(N_GPU_LAYERS),
    "--chat_format", "gemma",
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for server to start
print("Запуск сервера...")
time.sleep(10)

import requests
for i in range(30):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200:
            print(f"✓ llama.cpp сервер запущен на порту {PORT}")
            print(f"  Модель: {MODEL_FILE}")
            print(f"  Контекст: {N_CTX}")
            print(f"  GPU layers: {N_GPU_LAYERS}")
            break
    except:
        time.sleep(2)
else:
    print("✗ Сервер не запустился. Проверьте логи:")
    print(proc.stderr.read(2000) if proc.stderr else "No stderr")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, time

process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

tunnel_url = None
for _ in range(30):
    line = process.stderr.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"✓ Туннель активен!")
    print(f"\n{'='*60}")
    print(f"  URL: {tunnel_url}")
    print(f"  Health: {tunnel_url}/health")
    print(f"  API: {tunnel_url}/v1/chat/completions")
    print(f"{'='*60}")
else:
    print("✗ Не удалось получить URL туннеля")

In [ ]:
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")

response = client.chat.completions.create(
    model="gemma",
    messages=[
        {"role": "system", "content": """Extract from the invoice text: country, doc_type (electricity/telecom/bank/water/gas/tax/other), company, year.
Respond ONLY with JSON: {"country": "...", "doc_type": "...", "company": "...", "year": ...}"""},
        {"role": "user", "content": """RECHNUNG
Wiener Netze GmbH
Erdbergstraße 236, 1110 Wien
Rechnungsdatum: 15.03.2024
Stromrechnung für den Zeitraum 01.01.2024 - 31.03.2024
Gesamtbetrag: EUR 245,67 inkl. MwSt
UID: ATU12345678"""}
    ],
    temperature=0.0,
    max_tokens=200
)

print("Ответ модели:")
print(response.choices[0].message.content)

## Подключение к InvoiceLLM

Добавьте в `config.yaml`:

```yaml
llm:
  servers:
    - name: "Colab llama.cpp"
      host: "YOUR-URL.trycloudflare.com"
      port: 443
      ssl: true
      priority: 1
```

### Параметры для тонкой настройки

| Параметр | Описание | По умолчанию |
|----------|----------|-------------|
| `n_ctx` | Размер контекста (токены) | 4096 |
| `n_gpu_layers` | Слоёв на GPU (-1 = все) | 99 |
| `n_batch` | Размер батча | 512 |
| `n_threads` | CPU потоки | 4 |

### Другие модели

Замените `MODEL_REPO` и `MODEL_FILE` в ячейке настроек:
- `TheBloke/Llama-2-7B-Chat-GGUF` → `llama-2-7b-chat.Q4_K_M.gguf`
- `TheBloke/Mistral-7B-Instruct-v0.2-GGUF` → `mistral-7b-instruct-v0.2.Q4_K_M.gguf`